# UPI Pulse India: Digital Payments and UPI Transaction Trend Analysis in India

**AICTE | IBM SkillsBuild Data Analytics with AI Internship 2026**  
**Author:** Mo Faishal Haque  
**Dataset:** NPCI Monthly UPI Statistics (Demo/Official CSV)  
**Source:** https://www.npci.org.in/product/upi/product-statistics

---

## Project Overview

This notebook documents the complete data analytics workflow for the UPI Pulse India project.
It covers:
1. Data loading and validation
2. Exploratory Data Analysis (EDA)
3. KPI calculations
4. Trend and growth analysis
5. Anomaly detection
6. Visualisations
7. Forecasting (baseline linear model)
8. Findings and limitations

> **Note:** The demo dataset (`upi_transactions_sample.csv`) contains illustrative synthetic figures.
> It is labelled DEMO DATA throughout. Replace it with the official NPCI CSV for certified analysis.

## 1. Setup & Imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

# Try plotly for interactive charts, fall back to matplotlib
try:
    import plotly.express as px
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    PLOTLY = True
except ImportError:
    PLOTLY = False
    print('Plotly not installed; using matplotlib only.')

print('Setup complete.')
print(f'Pandas: {pd.__version__}, NumPy: {np.__version__}')

## 2. Data Loading & Validation

In [ ]:
# Load data directly (without Streamlit)
from pathlib import Path

DATA_DIR = Path('../data')
OFFICIAL_PATH = DATA_DIR / 'upi_transactions_official.csv'
SAMPLE_PATH   = DATA_DIR / 'upi_transactions_sample.csv'

if OFFICIAL_PATH.exists():
    df_raw = pd.read_csv(OFFICIAL_PATH)
    data_label = 'Official NPCI Data'
else:
    df_raw = pd.read_csv(SAMPLE_PATH)
    data_label = '⚠️ DEMO DATA (illustrative only)'

print(f'Data source: {data_label}')
print(f'Shape: {df_raw.shape}')
df_raw.head(10)

In [ ]:
# Data types and basic info
df_raw.info()

In [ ]:
# Missing values
print('Missing values per column:')
print(df_raw.isnull().sum())
print()
print('Duplicate rows:', df_raw.duplicated().sum())

In [ ]:
# Use the validated data_loader
from src.data_loader import load_data, get_annual_summary

df, meta = load_data()

print('=== Data Quality Report ===')
print(f"Source: {meta['source']}")
print(f"Demo data: {meta['is_demo']}")
print(f"Date range: {meta['date_range']}")
print(f"Raw rows: {meta['rows_raw']}  |  Clean rows: {meta['rows_clean']}")
if meta['warnings']:
    print('Warnings:')
    for w in meta['warnings']:
        print(f'  • {w}')
else:
    print('No data quality warnings.')

df.head()

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Descriptive statistics
df[['transaction_volume', 'transaction_value', 'avg_transaction_value']].describe().round(3)

In [ ]:
# Year distribution
print('Months per year:')
print(df.groupby('year').size())

In [ ]:
# Correlation matrix
corr = df[['transaction_volume', 'transaction_value', 'avg_transaction_value']].corr()
print('Correlation Matrix:')
corr.round(4)

## 4. KPI Calculations

In [ ]:
from src.analytics import (
    total_transaction_volume,
    total_transaction_value,
    average_transaction_value,
    period_over_period_growth,
    latest_period,
    highest_volume_period,
    lowest_volume_period,
    trend_summary,
)

tot_vol  = total_transaction_volume(df)
tot_val  = total_transaction_value(df)
avg_val  = average_transaction_value(df)
mom_vol  = period_over_period_growth(df, 'transaction_volume', 1)
yoy_vol  = period_over_period_growth(df, 'transaction_volume', 12)
best     = highest_volume_period(df)
worst    = lowest_volume_period(df)
lp       = latest_period(df)

print('=== KEY PERFORMANCE INDICATORS ===')
print(f'Dataset: {data_label}')
print()
print(f'Total Transaction Volume  : {tot_vol:>12,.2f} crore')
print(f'Total Transaction Value   : ₹{tot_val:>11,.3f} lakh crore')
print(f'Average Transaction Value : ₹{avg_val:>11,.0f} per transaction')
print(f'MoM Volume Growth (last)  : {mom_vol:>+11.1f} %')
print(f'YoY Volume Growth (last)  : {yoy_vol:>+11.1f} %')
print(f'Peak Month                : {best["period"]:>15} ({best["value"]:,.2f} Cr)')
print(f'Lowest Month              : {worst["period"]:>15} ({worst["value"]:,.2f} Cr)')
print(f'Latest Period             : {lp}')

### 4.1 Unit Verification

It is critical to use consistent units:

| Metric | Unit in CSV | SI Equivalent |
|--------|------------|---------------|
| transaction_volume | Crore | 1 Cr = 10,000,000 |
| transaction_value | ₹ Lakh Crore | 1 Lakh Cr = ₹1,000,000,000,000 (1 trillion) |
| avg_transaction_value | ₹ | Direct rupees |

Average per-transaction value formula:
```
avg = (transaction_value × 1e12) / (transaction_volume × 1e7)
```

In [ ]:
# Verify unit calculation on a single row
sample_row = df.iloc[-1]
vol_units   = sample_row['transaction_volume'] * 1e7   # transactions
val_rupees  = sample_row['transaction_value']  * 1e12  # rupees
avg_manual  = val_rupees / vol_units

print(f"Period     : {sample_row['period']}")
print(f"Volume     : {sample_row['transaction_volume']:,.2f} Cr = {vol_units:,.0f} transactions")
print(f"Value      : ₹{sample_row['transaction_value']:,.3f} lakh crore = ₹{val_rupees:,.0f}")
print(f"Avg manual : ₹{avg_manual:,.0f}")
print(f"Avg stored : ₹{sample_row['avg_transaction_value']:,.0f}")
print(f"Match: {abs(avg_manual - sample_row['avg_transaction_value']) < 1}")

## 5. Trend & Growth Analysis

In [ ]:
# Annual summary
annual = get_annual_summary(df)
annual[['year','total_volume','total_value','avg_transaction_value','yoy_volume_growth','yoy_value_growth']].round(2)

In [ ]:
# Monthly growth rates
growth_df = df[['period','transaction_volume','volume_mom_growth','volume_yoy_growth']].dropna(subset=['volume_mom_growth'])
print('Top 5 highest MoM growth months:')
print(growth_df.nlargest(5, 'volume_mom_growth')[['period','transaction_volume','volume_mom_growth']].to_string(index=False))
print()
print('Top 5 lowest MoM growth months:')
print(growth_df.nsmallest(5, 'volume_mom_growth')[['period','transaction_volume','volume_mom_growth']].to_string(index=False))

In [ ]:
# Trend summary
ts = trend_summary(df)
print('=== Trend Summary ===')
for k, v in ts.items():
    print(f'{k:35}: {v}')

## 6. Anomaly Detection

In [ ]:
from src.analytics import detect_anomalies

anomalies = detect_anomalies(df, 'transaction_volume', z_threshold=2.0)
print(f'Anomalous periods (z > 2.0): {len(anomalies)}')
if not anomalies.empty:
    print(anomalies[['period','transaction_volume','volume_mom_growth','anomaly_z_score','anomaly_reason']].to_string(index=False))

## 7. Visualisations

In [ ]:
# 7.1 Monthly Transaction Volume
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df['period'], df['transaction_volume'], color='#4c9be8', linewidth=1.5, marker='o', markersize=2)
ax.set_title('Monthly UPI Transaction Volume (Crore)', fontsize=13)
ax.set_xlabel('Period')
ax.set_ylabel('Volume (Crore)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.grid(alpha=0.3)
tick_step = max(1, len(df) // 20)
ax.set_xticks(range(0, len(df), tick_step))
ax.set_xticklabels(df['period'].iloc[::tick_step], rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.savefig('../assets/chart_volume_trend.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'[{data_label}]')

In [ ]:
# 7.2 Monthly Transaction Value
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df['period'], df['transaction_value'], color='#3fb950', linewidth=1.5, marker='o', markersize=2)
ax.set_title('Monthly UPI Transaction Value (₹ Lakh Crore)', fontsize=13)
ax.set_xlabel('Period')
ax.set_ylabel('Value (₹ Lakh Crore)')
ax.grid(alpha=0.3)
ax.set_xticks(range(0, len(df), tick_step))
ax.set_xticklabels(df['period'].iloc[::tick_step], rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.savefig('../assets/chart_value_trend.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# 7.3 Average Transaction Value
df2 = df.dropna(subset=['avg_transaction_value'])
fig, ax = plt.subplots(figsize=(14, 4))
ax.fill_between(range(len(df2)), df2['avg_transaction_value'], alpha=0.3, color='#d2a8ff')
ax.plot(range(len(df2)), df2['avg_transaction_value'], color='#d2a8ff', linewidth=1.5)
ax.set_title('Average Transaction Value per UPI Transfer (₹)', fontsize=13)
ax.set_ylabel('Avg Value (₹)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x:,.0f}'))
ax.set_xticks(range(0, len(df2), max(1, len(df2)//20)))
ax.set_xticklabels(df2['period'].iloc[::max(1, len(df2)//20)], rotation=45, ha='right', fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../assets/chart_avg_txn_value.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# 7.4 Year-over-Year Growth
yoy_df = df.dropna(subset=['volume_yoy_growth'])
if not yoy_df.empty:
    colors = ['#3fb950' if v >= 0 else '#f85149' for v in yoy_df['volume_yoy_growth']]
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.bar(range(len(yoy_df)), yoy_df['volume_yoy_growth'], color=colors, width=0.7)
    ax.axhline(0, color='#888', linewidth=0.8, linestyle='--')
    ax.set_title('Year-over-Year Transaction Volume Growth (%)', fontsize=13)
    ax.set_ylabel('YoY Growth (%)')
    step = max(1, len(yoy_df)//20)
    ax.set_xticks(range(0, len(yoy_df), step))
    ax.set_xticklabels(yoy_df['period'].iloc[::step], rotation=45, ha='right', fontsize=8)
    ax.grid(alpha=0.2)
    plt.tight_layout()
    plt.savefig('../assets/chart_yoy_growth.png', dpi=120, bbox_inches='tight')
    plt.show()
else:
    print('Insufficient data for YoY chart (need ≥ 13 months).')

In [ ]:
# 7.5 Annual Summary Bar
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(annual['year'].astype(str), annual['total_volume'], color='#4c9be8')
axes[0].set_title('Annual Transaction Volume (Crore)')
axes[0].set_ylabel('Volume (Crore)')
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(annual['year'].astype(str), annual['total_value'], color='#3fb950')
axes[1].set_title('Annual Transaction Value (₹ Lakh Crore)')
axes[1].set_ylabel('Value (₹ Lakh Crore)')
axes[1].tick_params(axis='x', rotation=45)

plt.suptitle('Annual UPI Summary', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../assets/chart_annual_summary.png', dpi=120, bbox_inches='tight')
plt.show()

## 8. Forecasting (Linear Baseline)

In [ ]:
from src.analytics import simple_forecast

forecast_df = simple_forecast(df, periods_ahead=6, metric='transaction_volume')

if forecast_df is not None:
    print('6-Period Ahead Forecast (Linear Baseline):')
    print(forecast_df[['period','forecast','lower','upper']].to_string(index=False))
    print()
    print('Model:', forecast_df['model'].iloc[0])
    print()
    print('⚠️ DISCLAIMER: This is a simple linear trend extrapolation.')
    print('   It does not model seasonality or external events.')
    print('   Do not use for financial decisions.')
else:
    print('Forecasting skipped: fewer than 24 months of data in current selection.')

In [ ]:
# Visualise forecast
if forecast_df is not None:
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(df['period'], df['transaction_volume'], color='#4c9be8', linewidth=1.5, label='Historical')
    ax.plot(forecast_df['period'], forecast_df['forecast'], color='#ffa657', 
            linewidth=2, linestyle='--', marker='o', markersize=5, label='Forecast')
    ax.fill_between(forecast_df['period'], forecast_df['lower'], forecast_df['upper'],
                    alpha=0.2, color='#ffa657', label='±1 Residual Std')
    ax.set_title('UPI Transaction Volume: Historical + Linear Forecast (Illustrative)', fontsize=13)
    ax.set_ylabel('Volume (Crore)')
    ax.legend()
    ax.grid(alpha=0.3)
    tick_positions = list(range(0, len(df), max(1, len(df)//15)))
    all_periods = list(df['period']) + list(forecast_df['period'])
    ax.set_xticks(range(len(all_periods)))
    ax.set_xticklabels(all_periods, rotation=45, ha='right', fontsize=7)
    plt.tight_layout()
    plt.savefig('../assets/chart_forecast.png', dpi=120, bbox_inches='tight')
    plt.show()

## 9. Key Findings

All findings below are derived from executed code. Numerical values reflect the loaded dataset.

In [ ]:
from src.insights import generate_local_insights

insights = generate_local_insights(df, annual)

print('=== KPI SUMMARY ===')
print(insights.get('kpi_summary',''))
print()
print('=== TREND ANALYSIS ===')
print(insights.get('trend_narrative',''))
print()
print('=== GROWTH ANALYSIS ===')
print(insights.get('growth_analysis',''))
print()
print('=== ANOMALY DETECTION ===')
print(insights.get('anomaly_detection',''))

In [ ]:
print('=== EXECUTIVE SUMMARY ===')
# Strip markdown bold markers for plain-text display
import re
summary = insights.get('executive_summary', '')
summary_plain = re.sub(r'\*\*(.+?)\*\*', r'\1', summary)
print(summary_plain)

## 10. Limitations

1. **Demo data is synthetic.** The bundled sample CSV contains illustrative figures. For peer-reviewed analysis, use official NPCI data.
2. **No state-level breakdown.** Geographic granularity requires additional data from NPCI ecosystem reports.
3. **Naive forecasting.** The linear model ignores seasonality, policy interventions, and economic shocks.
4. **No payment-app breakdown.** PhonePe, Google Pay, Paytm share data is not included.
5. **NPCI data is not a live API.** Manual download is required.
6. **This is an educational project.** It is not suitable for commercial or investment decisions.

## 11. References

1. NPCI UPI Product Statistics — https://www.npci.org.in/product/upi/product-statistics
2. NPCI UPI Ecosystem Statistics — https://www.npci.org.in/what-we-do/upi/upi-ecosystem-statistics
3. Reserve Bank of India Payment System Indicators — https://www.rbi.org.in
4. Streamlit Documentation — https://docs.streamlit.io
5. Plotly Documentation — https://plotly.com/python
6. Pandas Documentation — https://pandas.pydata.org
7. Scikit-learn Documentation — https://scikit-learn.org

---
*This notebook is part of the UPI Pulse India project submitted for the AICTE | IBM SkillsBuild Data Analytics with AI Internship 2026. Author: Mo Faishal Haque.*